# 🤖 Huấn Luyện & So Sánh A/B PPO 9D Cho Robot Tự Cân Bằng (Sim-to-Real STM32)

**Cấu hình đồng bộ với Firmware `Project_self` (STM32L476RG):**
- **Chu kỳ điều khiển:** 200 Hz (`Ts = 5 ms`), Encoder cập nhật mỗi `20 ms` (50 Hz, 1400 CPR, bộ lọc vận tốc $N=10$).
- **Chuỗi truyền động:** Hệ số khuếch đại PPO `1.30x` → Bộ lọc PWM `0.60 × u_raw + 0.40 × u_prev` → Làm tròn số nguyên → Bù vùng chết mềm `80 PWM` (deadzone `|u| ≤ 2 → 0`).
- **Không gian quan sát 9D:** $[\theta_t, \dot{\theta}_t, x_t, v_t, \theta_{t-1}, \dot{\theta}_{t-1}, x_{t-1}, v_{t-1}, a_{t-1}]$.
- **Bộ đánh giá toàn diện 98 kịch bản:** `90` ca chuẩn `10s` (`±1°, ±3°, ±5°` | nhiễu `0, ±1, ±2 N` | pin `7.2–8.4 V`) + `2` ca `0.0°, ±1.5 N` (`10s`) + `6` ca chạy dài `20–30s` (báo riêng từng nhóm **Không nhiễu / Có nhiễu / Chạy dài**).
- **Cổng an toàn STM32:** Bản thắng A/B luôn được lưu riêng tại `ab_winner_policy.npz` và `rl_sb3_stm32_ab_winner.h`. Chỉ cập nhật `rl_sb3_stm32.h` và `best_policy.npz` nếu đạt tiêu chí giữ vị trí (`|θ| ≤ 0.5°, |x| ≤ 2 cm` trong 2s cuối).

---
### Bước 1: Lấy code mới nhất từ GitHub, cài đặt thư viện & định nghĩa `run_step` (dừng ngay khi lỗi)

In [ ]:
import os
import sys
import subprocess

def run_step(cmd, required_files=None):
    """Chạy lệnh đồng bộ và DỪNG NGAY nếu lệnh trả về lỗi hoặc thiếu file đầu ra bắt buộc."""
    print(f"\n>>> RUN: {' '.join(cmd)}", flush=True)
    proc = subprocess.run(cmd)
    if proc.returncode != 0:
        raise RuntimeError(f"Lệnh thất bại với mã lỗi {proc.returncode}: {' '.join(cmd)}")
    if required_files:
        missing = [f for f in required_files if not os.path.exists(f)]
        if missing:
            raise FileNotFoundError(f"Thiếu file kết quả sau bước chạy: {missing}")

if os.path.exists("configs"):
    run_step(["git", "fetch", "origin"])
    run_step(["git", "reset", "--hard", "origin/p1-balance-robot"])
elif os.path.exists("self_blancing"):
    os.chdir("self_blancing")
    run_step(["git", "fetch", "origin"])
    run_step(["git", "reset", "--hard", "origin/p1-balance-robot"])
else:
    run_step(["git", "clone", "-b", "p1-balance-robot", "https://github.com/nguyenthanhsang2306/self_blancing.git"])
    os.chdir("self_blancing")

run_step([
    sys.executable, "-m", "pip", "install", "-q",
    "gymnasium", "stable-baselines3", "torch", "matplotlib", "pyyaml", "tensorboard"
])
print(f"[Ready] Thư mục làm việc hiện tại: {os.getcwd()}")

---
### Bước 2: Train 2 phiên bản `w_pos = 0.15` và `w_pos = 0.30` (giữ nguyên lọc PWM 0.60/0.40)

- Mỗi lần train sẽ xóa checkpoint cũ cùng nhãn để **chỉ chọn checkpoint tốt nhất trong chính lần train đó**, lưu riêng `.npz`, `.h` và bảng điểm 98 ca `.md`.
- Nếu bản `0.15` lỗi hoặc thiếu file kết quả, quá trình sẽ **dừng ngay** trước khi chạy tiếp.

In [ ]:
run_step(
    [sys.executable, "train.py", "--config", "configs/default.yaml", "--w-pos", "0.15", "--tag", "wpos_015"],
    required_files=[
        "outputs/checkpoints/best_policy_wpos_015.npz",
        "outputs/firmware_headers/rl_sb3_stm32_wpos_015.h",
        "outputs/reports/eval_report_wpos_015.md",
    ],
)

run_step(
    [sys.executable, "train.py", "--config", "configs/default.yaml", "--w-pos", "0.30", "--tag", "wpos_030"],
    required_files=[
        "outputs/checkpoints/best_policy_wpos_030.npz",
        "outputs/firmware_headers/rl_sb3_stm32_wpos_030.h",
        "outputs/reports/eval_report_wpos_030.md",
    ],
)

---
### Bước 3: So sánh `w_pos = 0.15` vs `0.30` trên 98 kịch bản & Lưu bản thắng A/B

- Báo cáo tách rõ **Không nhiễu 10s (18 ca)**, **Có nhiễu 10s (74 ca, gồm `0.0°, ±1.5 N`)**, và **Chạy dài 20–30s (6 ca)**.
- Luôn lưu bản thắng cuộc sang tên riêng `outputs/checkpoints/ab_winner_policy.npz`, `outputs/firmware_headers/rl_sb3_stm32_ab_winner.h` và lưu bảng điểm vào `outputs/reports/ab_comparison_report.md`.
- Chỉ cập nhật `best_policy.npz` và `rl_sb3_stm32.h` nếu model đạt tiêu chí cổng STM32 (`real_robot`).

In [ ]:
run_step(
    [
        sys.executable, "inference.py",
        "--compare",
        "outputs/checkpoints/best_policy_wpos_015.npz",
        "outputs/checkpoints/best_policy_wpos_030.npz",
        "--export-winner",
        "--report-out", "outputs/reports/ab_comparison_report.md",
    ],
    required_files=[
        "outputs/checkpoints/ab_winner_policy.npz",
        "outputs/firmware_headers/rl_sb3_stm32_ab_winner.h",
        "outputs/reports/ab_comparison_report.md",
    ],
)

---
### Bước 4: Vẽ đồ thị với tên ảnh riêng cho cả 2 checkpoint (`wpos_015` và `wpos_030`)

Mỗi checkpoint và kịch bản lưu vào file `.png` riêng trong `outputs/figures/` để tránh ghi đè lẫn nhau (tự động dừng nếu thiếu checkpoint hoặc lệnh vẽ lỗi).

In [ ]:
from IPython.display import Image, display

plot_jobs = [
    ("w_pos = 0.15 | Góc +5.0° chuẩn 10s", "outputs/checkpoints/best_policy_wpos_015.npz", ["--angle", "5.0", "--duration", "10.0"], "outputs/figures/wpos_015_angle5_10s.png"),
    ("w_pos = 0.15 | Góc +5.0° chạy dài 25s", "outputs/checkpoints/best_policy_wpos_015.npz", ["--angle", "5.0", "--duration", "25.0"], "outputs/figures/wpos_015_angle5_drift25s.png"),
    ("w_pos = 0.15 | Góc 0.0° + cú đẩy 1.5 N (10s)", "outputs/checkpoints/best_policy_wpos_015.npz", ["--angle", "0.0", "--kick", "1.5", "--duration", "10.0"], "outputs/figures/wpos_015_angle0_kick1p5N.png"),
    ("w_pos = 0.30 | Góc +5.0° chuẩn 10s", "outputs/checkpoints/best_policy_wpos_030.npz", ["--angle", "5.0", "--duration", "10.0"], "outputs/figures/wpos_030_angle5_10s.png"),
    ("w_pos = 0.30 | Góc +5.0° chạy dài 25s", "outputs/checkpoints/best_policy_wpos_030.npz", ["--angle", "5.0", "--duration", "25.0"], "outputs/figures/wpos_030_angle5_drift25s.png"),
    ("w_pos = 0.30 | Góc 0.0° + cú đẩy 1.5 N (10s)", "outputs/checkpoints/best_policy_wpos_030.npz", ["--angle", "0.0", "--kick", "1.5", "--duration", "10.0"], "outputs/figures/wpos_030_angle0_kick1p5N.png"),
]

for title, ckpt, extra_args, fig_out in plot_jobs:
    if not os.path.exists(ckpt):
        raise FileNotFoundError(f"Dừng vẽ đồ thị vì thiếu checkpoint: {ckpt}")
    print(f"\n=== {title} ===", flush=True)
    run_step(
        [sys.executable, "inference.py", "--weights", ckpt, *extra_args, "--output-figure", fig_out],
        required_files=[fig_out],
    )
    display(Image(filename=fig_out))

---
### Bước 5: Kiểm tra sai số toán học xuất C Header (`PyTorch ↔ Numpy ↔ C`)

In [ ]:
run_step([sys.executable, "verify_export.py"])

---
### Bước 6: Kiểm tra đủ kết quả cả 2 bản, đóng gói `rl_ab_comparison_bundle.zip` & Tải về

Gói tải về gồm đầy đủ:
- Checkpoint `best_policy_wpos_015.npz`, `best_policy_wpos_030.npz`, `ab_winner_policy.npz`
- Cấu hình `configs/default.yaml`
- Bảng điểm so sánh `outputs/reports/ab_comparison_report.md` và báo cáo từng bản
- Toàn bộ 6 ảnh đồ thị trong `outputs/figures/` và các header trong `outputs/firmware_headers/`

In [ ]:
import zipfile

required_bundle_files = [
    "outputs/checkpoints/best_policy_wpos_015.npz",
    "outputs/checkpoints/best_policy_wpos_030.npz",
    "outputs/checkpoints/ab_winner_policy.npz",
    "configs/default.yaml",
    "outputs/reports/ab_comparison_report.md",
    "outputs/reports/eval_report_wpos_015.md",
    "outputs/reports/eval_report_wpos_030.md",
    "outputs/firmware_headers/rl_sb3_stm32_wpos_015.h",
    "outputs/firmware_headers/rl_sb3_stm32_wpos_030.h",
    "outputs/firmware_headers/rl_sb3_stm32_ab_winner.h",
    "outputs/figures/wpos_015_angle5_10s.png",
    "outputs/figures/wpos_015_angle5_drift25s.png",
    "outputs/figures/wpos_015_angle0_kick1p5N.png",
    "outputs/figures/wpos_030_angle5_10s.png",
    "outputs/figures/wpos_030_angle5_drift25s.png",
    "outputs/figures/wpos_030_angle0_kick1p5N.png",
]

missing_files = [p for p in required_bundle_files if not os.path.exists(p)]
if missing_files:
    raise FileNotFoundError(f"Dừng đóng gói vì thiếu file kết quả: {missing_files}")

optional_files = ["outputs/firmware_headers/rl_sb3_stm32.h"]
bundle_files = required_bundle_files + [p for p in optional_files if os.path.exists(p)]

zip_name = "rl_ab_comparison_bundle.zip"
with zipfile.ZipFile(zip_name, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for fpath in bundle_files:
        zf.write(fpath, arcname=fpath)
        print(f"  + {fpath}")

print(f"\n[Bundle] Đã đóng gói thành công {len(bundle_files)} file vào {zip_name}")

try:
    from google.colab import files
    files.download(zip_name)
    files.download("outputs/reports/ab_comparison_report.md")
except Exception as e:
    print(f"Tự động tải bị chặn ({e}). Hãy tải thủ công từ cột trái Colab: {zip_name}")